## **Features de Atraso**

In [0]:
WITH pedidos_unicos AS (
    SELECT DISTINCT
        oi.seller_id,
        o.order_id,
        o.order_purchase_timestamp,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date
    FROM workspace.default.olist_orders_dataset o
    JOIN workspace.default.olist_order_items_dataset oi
        ON o.order_id = oi.order_id
    WHERE o.order_purchase_timestamp < DATE('2018-07-01')
),

tb AS (
    SELECT
        seller_id,
        order_id,
        order_purchase_timestamp,

        CASE
            WHEN order_delivered_customer_date > order_estimated_delivery_date
            THEN 1 ELSE 0
        END AS atrasado,

        DATEDIFF(
            order_delivered_customer_date,
            order_estimated_delivery_date
        ) AS dias_atraso,

        CASE WHEN order_purchase_timestamp >= DATE('2018-07-01') - INTERVAL 28 DAYS THEN 1 END AS flag_28d,
        CASE WHEN order_purchase_timestamp >= DATE('2018-07-01') - INTERVAL 56 DAYS THEN 1 END AS flag_56d,
        CASE WHEN order_purchase_timestamp >= DATE('2018-07-01') - INTERVAL 365 DAYS THEN 1 END AS flag_365d
    FROM pedidos_unicos
)

SELECT
    seller_id,

    SUM(atrasado) AS n_atrasados_vida,
    SUM(CASE WHEN flag_28d = 1 THEN atrasado END) AS n_atrasados_28d,
    SUM(CASE WHEN flag_56d = 1 THEN atrasado END) AS n_atrasados_56d,
    SUM(CASE WHEN flag_365d = 1 THEN atrasado END) AS n_atrasados_365d,

    100.0 * SUM(atrasado) / COUNT(*) AS pct_atrasados_vida,

    100.0 * SUM(CASE WHEN flag_28d = 1 THEN atrasado END)
           / NULLIF(SUM(flag_28d), 0) AS pct_atrasados_28d,

    100.0 * SUM(CASE WHEN flag_56d = 1 THEN atrasado END)
           / NULLIF(SUM(flag_56d), 0) AS pct_atrasados_56d,

    100.0 * SUM(CASE WHEN flag_365d = 1 THEN atrasado END)
           / NULLIF(SUM(flag_365d), 0) AS pct_atrasados_365d,

    AVG(CASE WHEN atrasado = 1 THEN dias_atraso END) AS atraso_medio_vida,

    AVG(CASE WHEN atrasado = 1 AND flag_28d = 1 THEN dias_atraso END) AS atraso_medio_28d,
    AVG(CASE WHEN atrasado = 1 AND flag_56d = 1 THEN dias_atraso END) AS atraso_medio_56d,
    AVG(CASE WHEN atrasado = 1 AND flag_365d = 1 THEN dias_atraso END) AS atraso_medio_365d

FROM tb
GROUP BY seller_id
ORDER BY seller_id;
